Make J-plot from GOES/SUVI images to track the speed of a solar feature

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator
from scipy.interpolate import UnivariateSpline
import astropy.io.fits as fits
import astropy.units as u
from astropy.coordinates import SkyCoord
import sunpy.map
from sunkit_instruments import suvi
from scipy import stats
import sunpy.sun.constants as const
from copy import deepcopy
from scipy.ndimage import gaussian_filter
from PIL import Image
import matplotlib.colors as colors
from astropy.visualization import ImageNormalize, LogStretch, LogStretch, PercentileInterval

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 100
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

data_dir = '/home/mnedal/data'
savedir = '/home/mnedal/repos/dias_work'

In [ ]:
os.makedirs(f'{savedir}/jplots/suvi/bezier', exist_ok=True)

In [ ]:
def split_datetime(start=None, end=None):
    
    START_DATE, START_TIME = start.split('T')
    END_DATE, END_TIME = end.split('T')

    START_YEAR, START_MONTH, START_DAY = START_DATE.split('-')
    END_YEAR, END_MONTH, END_DAY = END_DATE.split('-')

    START_HOUR, START_MINUTE, START_SECOND = START_TIME.split(':')
    END_HOUR, END_MINUTE, END_SECOND = END_TIME.split(':')

    datetime_dict = {
        'start_year': START_YEAR,
        'start_month': START_MONTH,
        'start_day': START_DAY,
        'start_hour': START_HOUR,
        'start_minute': START_MINUTE,
        'start_second': START_SECOND,
        
        'end_year': END_YEAR,
        'end_month': END_MONTH,
        'end_day': END_DAY,
        'end_hour': END_HOUR,
        'end_minute': END_MINUTE,
        'end_second': END_SECOND
    }
    return datetime_dict



def plot_line(angle_deg=None, length=None, map_obj=None):
    """
    Plot a straight line at an angle in degrees from the solar West.
    """
    angle_rad = np.deg2rad(angle_deg)

    # Define the length of the line (in arcseconds)
    line_length = length * u.arcsec

    # Define the center point of the line (e.g., the center of the map)
    center = map_obj.center

    # Calculate the start and end points of the line
    start_point = SkyCoord(center.Tx, center.Ty, frame=map_obj.coordinate_frame)

    end_point = SkyCoord(center.Tx + line_length * np.cos(angle_rad),
                        center.Ty + line_length * np.sin(angle_rad),
                        frame=map_obj.coordinate_frame)
    
    line = SkyCoord([start_point, end_point])
    return line



def load_suvi(start=None, end=None, channel=195):
    """
    * 9.4 nm (FeXVIII)
    * 13.1 nm (FeXXI)
    * 17.1 nm (FeIX/X)
    * 19.5 nm (FeXII)
    * 28.4 nm (FeXV)
    * 30.4 nm (HeII)
    """
    dt_dict = split_datetime(start=start, end=end)
    data_path = f"{data_dir}/SUVI/{dt_dict['start_year']}{dt_dict['start_month']}{dt_dict['start_day']}/{channel}A"
    root_filename = f"dr_suvi-l2-ci{channel}_g18_s"
    data = sorted(glob.glob(f"{data_path}/{root_filename}{dt_dict['start_year']}{dt_dict['start_month']}{dt_dict['start_day']}*.fits"))
    
    start_file_to_find = f"{data_path}/{root_filename}{dt_dict['start_year']}{dt_dict['start_month']}{dt_dict['start_day']}T{dt_dict['start_hour']}0000Z_e{dt_dict['start_year']}{dt_dict['start_month']}{dt_dict['start_day']}T{dt_dict['start_hour']}0400Z_v1-0-2.fits"
    end_file_to_find   = f"{data_path}/{root_filename}{dt_dict['end_year']}{dt_dict['end_month']}{dt_dict['end_day']}T{dt_dict['end_hour']}2800Z_e{dt_dict['end_year']}{dt_dict['end_month']}{dt_dict['end_day']}T{dt_dict['end_hour']}3200Z_v1-0-2.fits"
    
    idx1 = data.index(start_file_to_find)
    idx2 = data.index(end_file_to_find)
    
    chosen_files = data[idx1:idx2]
    
    map_objects = []
    for i, file in enumerate(chosen_files):
        m = suvi.files_to_map(file, despike_l1b=True)
        min_range = 0
        if channel == 94:
            max_range = 20
        elif channel == 171:
            max_range = 20
        elif channel == 131:
            max_range = 20
        elif channel == 195:
            max_range = 50
        elif channel == 284:
            max_range = 50
        elif channel == 304:
            max_range = 100
        
        m.plot_settings['norm'] = ImageNormalize(vmin=min_range, vmax=max_range, stretch=LogStretch())
        map_objects.append(m)
        print(f'SUVI image {i} is done')
    return map_objects



def generate_centered_list(center, difference, num_elements):
    """
    Generate a list of numbers centered around a given number with a specified difference
    between consecutive numbers.

    Parameters:
    center (int): The central number around which the list is generated.
    difference (int): The difference between consecutive numbers in the list.
    num_elements (int): The number of elements before and after the central number.

    Returns:
    list: A list of numbers centered around the specified central number.
    """
    return [center + difference * i for i in range(-num_elements, num_elements + 1)]



def remove_redundant_maps(maps):
    """
    Remove redundant SunPy maps, keeping only one map per unique timestamp.

    Parameters:
    maps (list): List of SunPy Map objects. Each map is expected to have a 'date-obs' 
                 key in its metadata that provides the observation timestamp.

    Returns:
    list: A list of unique SunPy Map objects, one per unique timestamp.
    
    Example:
    >>> unique_maps = remove_redundant_maps(list_of_sunpy_maps)
    """
    unique_maps = {}
    for m in maps:
        timestamp = m.latex_name
        if timestamp not in unique_maps:
            unique_maps[timestamp] = m
    return list(unique_maps.values())



def apply_runratio(maps):
    """
    Apply running-ratio image technique on EUV images.
    See: https://iopscience.iop.org/article/10.1088/0004-637X/750/2/134/pdf
        Inputs: list of EUV sunpy maps.
        Output: sequence of run-ratio sunpy maps.
    """
    runratio = [m / prev_m.quantity for m, prev_m in zip(maps[1:], maps[:-1])]
    m_seq_runratio = sunpy.map.Map(runratio, sequence=True)
    
    for m in m_seq_runratio:
        m.data[np.isnan(m.data)] = 1
        m.plot_settings['norm'] = colors.Normalize(vmin=0, vmax=2)
        m.plot_settings['cmap'] = 'Greys_r'
    
    return m_seq_runratio



def enhance_contrast(image, vmin, vmax):
    """
    Enhance contrast by clipping and normalization.
    """
    image_clipped = np.clip(image, vmin, vmax)
    image_normalized = (image_clipped - vmin) / (vmax - vmin)
    return image_normalized



def calculate_percentiles(image, lower_percentile=3, upper_percentile=97):
    """
    Calculate vmin and vmax based on the 1st and 99th percentiles.
    """
    vmin = np.percentile(image, lower_percentile)
    vmax = np.percentile(image, upper_percentile)
    return vmin, vmax



def round_obstime(time=None):
    """
    Round the observation time to put it in the image title.
    Input : str, time (HH:MM:SS.sss)
    Output: str, datetime (YYYY-mm-DD HH:MM:SS)
    """
    from datetime import datetime, timedelta

    original_time_str = time

    # Convert the original time string to a datetime object
    original_time = datetime.strptime(original_time_str, '%H:%M:%S.%f')

    # Round the time to the nearest second
    rounded_time = original_time + timedelta(seconds=round(original_time.microsecond / 1e6))

    # Format the rounded time as 'HH:MM:SS'
    rounded_time_str = rounded_time.strftime('%H:%M:%S')
    
    return rounded_time_str



def draw_bezier(x1=0, y1=0, x2=0, y2=0, control=[0,0]):
    """
    Draw a Bezier curve using the given control points.
    The curve will be drawn from the point (x1, y1) to the point
    (x2, y2) using the control points (control[0], control[1]).
    """
    A = np.array([x2, y2])
    B = np.array(control)
    C = np.array([x1, y1])

    A = A.reshape(2,1)
    B = B.reshape(2,1)
    C = C.reshape(2,1)
    
    t = np.arange(0, 1, 0.2).reshape(1,-1)
    
    P0 = A * t + (1 - t) * B
    P1 = B * t + (1 - t) * C
    Pfinal = P0 * t + (1 - t) * P1

    return Pfinal


def extract_bezier_values(array, x1, y1, x2, y2, dx, control):
    """
    Extract the values of a Bezier curve at the given control points.
    The curve will be drawn from the point (x1, y1) to the point
    (x2, y2) using the control points (control[0], control[1])
    """
    Pfinal = draw_bezier(x1, y1, x2, y2, dx, control)
    x_coords = np.round(Pfinal[0, :]).astype(int)
    y_coords = np.round(Pfinal[1, :]).astype(int)

    # Clip the coordinates to stay within array bounds
    x_coords = np.clip(x_coords, 0, array.shape[1] - 1)
    y_coords = np.clip(y_coords, 0, array.shape[0] - 1)

    # Extract values along the Bézier curve
    bezier_values = array[y_coords, x_coords]
    
    return bezier_values, x_coords, y_coords



def generate_number_list(center, offset, count):
    """
    Generate a list of numbers around a given center number.

    Parameters:
    center (int or float): The central number of the list.
    offset (int or float): The increment by which numbers are spaced from the center.
    count (int): The number of numbers to include on each side of the center number.

    Returns:
    list: A list of numbers around the given center.

    Example usage:
        center_number = 0
        offset_value = 2
        count_around = 3
        number_list = generate_number_list(center_number, offset_value, count_around)
        print(number_list)
        [-6, -4, -2, 0, 2, 4, 6]
    """
    return [center + offset * i for i in range(-count, count + 1)]



def compute_standard_error(values_list):
    values_array = np.array(values_list)
    mean_values = np.mean(values_array, axis=0)
    standard_error = np.std(values_array, axis=0, ddof=1) / np.sqrt(values_array.shape[0])
    return mean_values, standard_error

In [ ]:
suvi_171_map_objects = load_suvi(start='2024-05-14T17:00:00', end='2024-05-14T18:00:00', channel=171)
suvi_195_map_objects = load_suvi(start='2024-05-14T17:00:00', end='2024-05-14T18:00:00', channel=195)
suvi_284_map_objects = load_suvi(start='2024-05-14T17:00:00', end='2024-05-14T18:00:00', channel=284)

In [ ]:
print(len(suvi_171_map_objects), len(suvi_195_map_objects), len(suvi_284_map_objects))

In [ ]:
index = 0

fig = plt.figure(figsize=[15,5])

ax = fig.add_subplot(131, projection=suvi_171_map_objects[index])
suvi_171_map_objects[index].plot(axes=ax)
ax.grid(False)

ax = fig.add_subplot(132, projection=suvi_195_map_objects[index])
suvi_195_map_objects[index].plot(axes=ax)
ax.grid(False)

ax = fig.add_subplot(133, projection=suvi_284_map_objects[index])
suvi_284_map_objects[index].plot(axes=ax)
ax.grid(False)

fig.tight_layout()
plt.show()

In [ ]:
clean_maps_171A = remove_redundant_maps(suvi_171_map_objects)
clean_maps_195A = remove_redundant_maps(suvi_195_map_objects)
clean_maps_284A = remove_redundant_maps(suvi_284_map_objects)

print(len(clean_maps_171A), len(clean_maps_195A), len(clean_maps_284A))

In [ ]:
# make run-diff maps
m_seq_runratio_171A = apply_runratio(clean_maps_171A)
m_seq_runratio_195A = apply_runratio(clean_maps_195A)
m_seq_runratio_284A = apply_runratio(clean_maps_284A)

print(len(m_seq_runratio_171A), len(m_seq_runratio_195A), len(m_seq_runratio_284A))

In [ ]:
rgb_image = np.stack([m_seq_runratio_171A[6].data,
                      m_seq_runratio_195A[6].data,
                      m_seq_runratio_284A[6].data],
                     axis=-1)

# Enhance contrast for each channel
rgb_image_enhanced = np.zeros_like(rgb_image)
for i in range(3):  # Process each channel independently
    vmin, vmax = calculate_percentiles(rgb_image[..., i])
    rgb_image_enhanced[..., i] = enhance_contrast(rgb_image[..., i], vmin, vmax)

In [ ]:
m = m_seq_runratio_171A[6]

slits = True
centered_list = generate_centered_list(160, 2, 6)

fig = plt.figure(figsize=[7,7])
ax = fig.add_subplot(111, projection=m)
ax.imshow(rgb_image_enhanced, vmin=0, vmax=1)

date_str = str(m.date).split('T')[0]
rounded_time_str = round_obstime(time=str(m.date).split('T')[1])
ax.set_title(f'SUVI 171, 195, 284 $\AA$ {date_str} {rounded_time_str}')
ax.set_xlabel('Helioprojective Longitude (arcsec)')
ax.set_ylabel('Helioprojective Latitude (arcsec)')

if slits:
    for value in centered_list:
        line = plot_line(angle_deg=value, length=1870, map_obj=m)
        ax.plot_coord(line, color='black')
        
        # Plot the number at the end of the line
        # Convert SkyCoord to pixel coordinates for plotting text
        line_lon, line_lat = line.Tx, line.Ty
        end_point_pixel = m.world_to_pixel(SkyCoord(line_lon[1], line_lat[1], frame=m.coordinate_frame))
        
        # Display the number at the end point
        ax.text(end_point_pixel.x.value - 20, end_point_pixel.y.value + 20, f'{value}$^o$',
                color='black', fontsize=10, ha='center', va='center')
plt.show()

In [ ]:
# Convert the 3D array to a 2D array by averaging the channels
array_2d = np.mean(rgb_image_enhanced, axis=2)
m = sunpy.map.Map(array_2d, m.meta)

In [ ]:
slits = True

fig = plt.figure(figsize=[7,7])
ax = fig.add_subplot(111, projection=m)
m.plot(axes=ax, cmap='Greys_r', vmin=0.2, vmax=2)
ax.set_title(f'SUVI 171, 195, 284 $\AA$ {date_str} {rounded_time_str}')

if slits:
    for value in centered_list:
        line = plot_line(angle_deg=value, length=1870, map_obj=m)
        ax.plot_coord(line, color='black')
        
        # Plot the number at the end of the line
        # Convert SkyCoord to pixel coordinates for plotting text
        line_lon, line_lat = line.Tx, line.Ty
        end_point_pixel = m.world_to_pixel(SkyCoord(line_lon[1], line_lat[1], frame=m.coordinate_frame))
        
        # Display the number at the end point
        ax.text(end_point_pixel.x.value - 20, end_point_pixel.y.value + 20, f'{value}$^o$',
                color='black', fontsize=10, ha='center', va='center')
plt.show()

### Inspect individual channel

In [ ]:
slits = False

m = m_seq_runratio_195A[6]
m.plot_settings['norm'] = colors.Normalize(vmin=0.7, vmax=1.3)

fig = plt.figure(figsize=[7,7])
ax = fig.add_subplot(111, projection=m)
m.plot(axes=ax, cmap='Greys_r')

if slits:
    for value in centered_list:
        line = plot_line(angle_deg=value, length=1870, map_obj=m)
        ax.plot_coord(line, color='black')
        
        # Plot the number at the end of the line
        # Convert SkyCoord to pixel coordinates for plotting text
        line_lon, line_lat = line.Tx, line.Ty
        end_point_pixel = m.world_to_pixel(SkyCoord(line_lon[1], line_lat[1], frame=m.coordinate_frame))
        
        # Display the number at the end point
        ax.text(end_point_pixel.x.value - 20, end_point_pixel.y.value + 20, f'{value}$^o$',
                color='black', fontsize=10, ha='center', va='center')
plt.show()

In [1]:



centered_list = generate_centered_list(160, 2, 6)


# For multuple time instances

# Make a dictionary to hold the lists of lists
intensity_dict = {}
distances_dict = {}
output_obj = {}
jmaps_coords_list = {}

# Initialize each key with an empty list
for value in centered_list:
    intensity_dict[f'intensity_{value}deg'] = []
    distances_dict[f'distances_{value}deg'] = []
    output_obj[f'intensity_{value}deg'] = []
    output_obj[f'distances_{value}deg'] = []
    jmaps_coords_list[f'{value}'] = []

output_obj['time'] = []
output_obj['map_obj'] = []
output_obj['instrument'] = []

# for i, m in enumerate(suvi_rgb):
for i, m in enumerate(rgb_maps):
    print(f'Working on map {i} ..')
    
    m.plot_settings['norm'] = colors.Normalize(vmin=0.7, vmax=1.3)
    
    fig = plt.figure(figsize=[7,7])
    ax = fig.add_subplot(111, projection=m)
    m.plot(axes=ax, cmap='Greys_r')#, vmin=0.2, vmax=2)
    
    for value in centered_list:
        line = plot_line(angle_deg=value, length=1870, map_obj=m)
        ax.plot_coord(line, color='black')
        
        # Plot the number at the end of the line
        # Convert SkyCoord to pixel coordinates for plotting text
        line_lon, line_lat = line.Tx, line.Ty
        end_point_pixel = m.world_to_pixel(SkyCoord(line_lon[1], line_lat[1], frame=m.coordinate_frame))
        
        # Display the number at the end point
        ax.text(end_point_pixel.x.value - 20, end_point_pixel.y.value + 20, f'{value}$^o$',
                color='black', fontsize=10, ha='center', va='center')
        
        # obtain the coordinates of the map pixels that intersect that path
        intensity_coords_slit = sunpy.map.pixelate_coord_path(m, line)
        
        # Create mask to identify valid coordinates
        valid_mask = sunpy.map.contains_coordinate(m, intensity_coords_slit)
        
        # Apply the mask to filter valid coordinates
        valid_coords = intensity_coords_slit[valid_mask]
        
        # Pass those coordinates to extract the values for those map pixels
        intensity_slit = sunpy.map.sample_at_coords(m, valid_coords)
        
        # Calculate the angular separation between the first point and every other coordinate we extracted
        angular_separation_slit = valid_coords.separation(valid_coords[0]).to(u.arcsec)
        
        # Append the values to the lists
        intensity_dict[f'intensity_{value}deg'].append(list(intensity_slit.value))
        distances_dict[f'distances_{value}deg'].append(list(angular_separation_slit.value))
    
    output_obj['time'].append(m.date.iso)
    output_obj['map_obj'].append(m)
    output_obj['instrument'].append(f"AIA {m.meta['wavelnth']}A")
    
    plt.show()













## J-Maps for SUVI

In [ ]:
suvi_rgb = []

for idx, _ in enumerate(m_seq_runratio_171A):
    
    rgb_image = np.stack([m_seq_runratio_171A[idx].data,
                          m_seq_runratio_195A[idx].data,
                          m_seq_runratio_284A[idx].data],
                         axis=-1)
    
    # Enhance contrast for each channel
    rgb_image_enhanced = np.zeros_like(rgb_image)
    for i in range(3):  # Process each channel independently
        vmin, vmax = calculate_percentiles(rgb_image[..., i])
        rgb_image_enhanced[..., i] = enhance_contrast(rgb_image[..., i], vmin, vmax)
    
    # Convert the 3D array to a 2D array by averaging the channels
    array_2d = np.mean(rgb_image_enhanced, axis=2)
    m = sunpy.map.Map(array_2d, m_seq_runratio_171A[idx].meta)
    suvi_rgb.append(m)

print('All SUVI RR images have been prepared as RGB-like sunpy maps', len(suvi_rgb))

In [ ]:
# Make a dictionary to hold the lists of lists
intensity_dict = {}
distances_dict = {}
output_obj = {}
jmaps_coords_list = {}

# Initialize each key with an empty list
for value in centered_list:
    intensity_dict[f'intensity_{value}deg'] = []
    distances_dict[f'distances_{value}deg'] = []
    output_obj[f'intensity_{value}deg'] = []
    output_obj[f'distances_{value}deg'] = []
    jmaps_coords_list[f'{value}'] = []

output_obj['time'] = []
output_obj['map_obj'] = []
output_obj['instrument'] = []

# for i, m in enumerate(suvi_rgb):
for i, m in enumerate(m_seq_runratio_195A):
    print(f'Working on map {i} ..')
    
    m.plot_settings['norm'] = colors.Normalize(vmin=0.7, vmax=1.3)
    
    fig = plt.figure(figsize=[7,7])
    ax = fig.add_subplot(111, projection=m)
    m.plot(axes=ax, cmap='Greys_r')#, vmin=0.2, vmax=2)
    
    for value in centered_list:
        line = plot_line(angle_deg=value, length=1870, map_obj=m)
        ax.plot_coord(line, color='black')
        
        # Plot the number at the end of the line
        # Convert SkyCoord to pixel coordinates for plotting text
        line_lon, line_lat = line.Tx, line.Ty
        end_point_pixel = m.world_to_pixel(SkyCoord(line_lon[1], line_lat[1], frame=m.coordinate_frame))
        
        # Display the number at the end point
        ax.text(end_point_pixel.x.value - 20, end_point_pixel.y.value + 20, f'{value}$^o$',
                color='black', fontsize=10, ha='center', va='center')
        
        # obtain the coordinates of the map pixels that intersect that path
        intensity_coords_slit = sunpy.map.pixelate_coord_path(m, line)
        
        # Create mask to identify valid coordinates
        valid_mask = sunpy.map.contains_coordinate(m, intensity_coords_slit)
        
        # Apply the mask to filter valid coordinates
        valid_coords = intensity_coords_slit[valid_mask]
        
        # Pass those coordinates to extract the values for those map pixels
        intensity_slit = sunpy.map.sample_at_coords(m, valid_coords)
        
        # Calculate the angular separation between the first point and every other coordinate we extracted
        angular_separation_slit = valid_coords.separation(valid_coords[0]).to(u.arcsec)
        
        # Append the values to the lists
        intensity_dict[f'intensity_{value}deg'].append(list(intensity_slit.value))
        distances_dict[f'distances_{value}deg'].append(list(angular_separation_slit.value))
    
    output_obj['time'].append(m.date.iso)
    output_obj['map_obj'].append(m)
    output_obj['instrument'].append(f"SUVI {m.meta['wavelnth']}A")
#     output_obj['instrument'].append('SUVI_171_195_284_A')
    
    plt.show()

In [ ]:
datenum_arr = [mdates.date2num(pd.Timestamp(str(t))) for t in output_obj['time']]

for value in centered_list:
    intens = np.array(intensity_dict[f'intensity_{value}deg']).T
    height = np.array(distances_dict[f'distances_{value}deg'][0])
    output_obj[f'intensity_{value}deg'].append(intens)
    output_obj[f'distances_{value}deg'].append(height)

In [ ]:
list(output_obj.keys())

In [ ]:
def find_nearest_index(array, value):
    # Check if the value exists in the array
    if value in array:
        return np.where(array == value)[0][0]  # Return the index of the value
    else:
        # Find the index of the nearest value
        nearest_index = (np.abs(array - value)).argmin()
        return nearest_index

In [ ]:
# ### Test ... 

# def draw_bezier(x1=0, y1=0, x2=0, y2=0, dx=0, control=[0,0]):
#     """
#     Draw a Bezier curve using the given control points.
#     The curve will be drawn from the point (x1, y1) to the point
#     (x2, y2) using the control points (control[0], control[1]).
#     """
#     A = np.array([x2, y2])
#     B = np.array(control)
#     C = np.array([x1, y1])
    
#     A = A.reshape(2,1)
#     B = B.reshape(2,1)
#     C = C.reshape(2,1)
    
#     t = np.arange(0, 1+dx, dx).reshape(1,-1)
    
#     P0 = A * t + (1 - t) * B
#     P1 = B * t + (1 - t) * C
#     Pfinal = P0 * t + (1 - t) * P1
    
#     return Pfinal



# angle = 168

# height = output_obj[f'distances_{angle}deg'][0].copy()
# intens = output_obj[f'intensity_{angle}deg'][0].copy()
# suvi_map = output_obj['map_obj'][0]
# event_date = suvi_map.meta['date'].split('T')[0]

# psudo_time = np.arange(len(datenum_arr))
# psudo_dist = np.arange(len(height))

# # Define start and end points and control point for the Bézier curve
# dx = 0.35
# dy = 40
# X_start = generate_number_list(5, dx, 2)
# X_end   = generate_number_list(7, dx, 2)
# Y_start = generate_number_list(1250, dy, 2)
# Y_end   = generate_number_list(1600, dy, 2)
# control = [7, 1600]

# x_coords_list = []
# y_coords_list = []

# fig = plt.figure(figsize=[8,6])
# ax = fig.add_subplot(111)
# ax.pcolormesh(psudo_time, height, intens, vmin=0.5, vmax=1.7, cmap='Greys_r')

# # Extract values along the Bézier curve
# for (x1, y1, x2, y2) in zip(X_start, Y_start, X_end, Y_end):
#     _, x_coords, y_coords = extract_bezier_values(intens, x1, y1, x2, y2, dx, control)
#     x_coords_list.append(x_coords[:-1])
#     y_coords_list.append(y_coords[:-1])
#     ax.plot(x_coords[:-1], y_coords[:-1], 'ko--', linewidth=1, markersize=4, alpha=0.5)

# # # Compute mean and standard error for x and y coordinates
# # mean_x_coords, standard_error_x = compute_standard_error(x_coords_list)
# # mean_y_coords, standard_error_y = compute_standard_error(y_coords_list)

# # # Plot the mean Bezier curve with error bars
# # ax.plot(mean_x_coords, mean_y_coords, 'ro--', linewidth=1, markersize=5)
# # ax.errorbar(mean_x_coords, mean_y_coords,
# #             xerr=standard_error_x, yerr=standard_error_y,
# #             fmt='ro', ecolor='blue', barsabove=False, markersize=4, elinewidth=1, capsize=3)

# ax.set_xlabel('Time step')
# ax.set_ylabel('Height step')
# ax.set_title(f"J-plot of {output_obj['instrument'][0]} Intensity Along Slit {angle} deg")
# ax.set_ylim(bottom=suvi_map.rsun_obs.value)
# ax.set_xlim(right=12)
# plt.show()

In [ ]:
x_coords_list

In [ ]:
y_coords_list

In [ ]:
centered_list

In [ ]:
def draw_bezier(x1=0, y1=0, x2=0, y2=0, dx=0, control=[0,0]):
    """
    Draw a Bezier curve using the given control points.
    The curve will be drawn from the point (x1, y1) to the point
    (x2, y2) using the control points (control[0], control[1]).
    """
    A = np.array([x2, y2])
    B = np.array(control)
    C = np.array([x1, y1])
    
    A = A.reshape(2,1)
    B = B.reshape(2,1)
    C = C.reshape(2,1)

    # dx = x2 - x1
    # t = np.linspace(0, 1, dx+1).reshape(1,-1)
    # t = np.arange(0, 1.25, 0.25).reshape(1,-1)
    # t = np.arange(0, 1.15, 0.15).reshape(1,-1)
    t = np.arange(0, 1+dx, dx).reshape(1,-1)
    
    P0 = A * t + (1 - t) * B
    P1 = B * t + (1 - t) * C
    Pfinal = P0 * t + (1 - t) * P1
    
    return Pfinal



angle = 168

height = output_obj[f'distances_{angle}deg'][0].copy()
intens = output_obj[f'intensity_{angle}deg'][0].copy()
suvi_map = output_obj['map_obj'][0]
event_date = suvi_map.meta['date'].split('T')[0]

psudo_time = np.arange(len(datenum_arr))
psudo_dist = np.arange(len(height))

# Define start and end points and control point for the Bézier curve
dx = 0.35
dy = 15
X_start = generate_number_list(5, dx, 2)
X_end   = generate_number_list(7, dx, 2)
Y_start = generate_number_list(600, dy, 2)
Y_end   = generate_number_list(880, dy, 2)
control = [7, 700]

x_coords_list = []
y_coords_list = []

fig = plt.figure(figsize=[8,6])
ax = fig.add_subplot(111)
ax.pcolormesh(psudo_time, psudo_dist, intens, vmin=0.5, vmax=1.7, cmap='Greys_r')

# Extract values along the Bézier curve
for (x1, y1, x2, y2) in zip(X_start, Y_start, X_end, Y_end):
    _, x_coords, y_coords = extract_bezier_values(intens, x1, y1, x2, y2, dx, control)
    x_coords_list.append(x_coords[:-1])
    y_coords_list.append(y_coords[:-1])
    ax.plot(x_coords[:-1], y_coords[:-1], 'ko--', linewidth=1, markersize=4, alpha=0.5)

# Compute mean and standard error for x and y coordinates
mean_x_coords, standard_error_x = compute_standard_error(x_coords_list)
mean_y_coords, standard_error_y = compute_standard_error(y_coords_list)

# Plot the mean Bezier curve with error bars
ax.plot(mean_x_coords, mean_y_coords, 'ro--', linewidth=1, markersize=5)
ax.errorbar(mean_x_coords, mean_y_coords,
            xerr=standard_error_x, yerr=standard_error_y,
            fmt='ro', ecolor='blue', barsabove=False, markersize=4, elinewidth=1, capsize=3)

ax.set_xlabel('Time step')
ax.set_ylabel('Height step')
ax.set_title(f"J-plot of {output_obj['instrument'][0]} Intensity Along Slit {angle} deg")
ax.yaxis.set_minor_locator(AutoMinorLocator(n=5))
sol_limb = find_nearest_index(height, suvi_map.rsun_obs.value)
# print(sol_limb)
ax.set_ylim(bottom=sol_limb)#, top=psudo_dist[-1])
ax.set_xlim(right=12)
plt.show()

In [ ]:
# # Define start and end points and control point for the Bézier curve
# xx_start = [4, 5, 5, 3, 3, 3, 2, 3, 3, 
# xx_end   = [8, 9, 9, 10, 10, 10, 8.5, 8, 8, 
# dx = [           0.25, 
# yy_start = [740, 600, 640, 600, 580, 550, 560, 570, 590, 
# yy_end   = [1200, 1100, 1000, 1100, 1050, 1050, 900, 900, , 
# controls = [ [6,1000], [7,970], [7,900], [6,810], [6,770], [6,770], [6,700], [5,680], 

In [ ]:
# Initialize an empty dictionary to store the data for each angle
angle_data = {}

# Loop over the list of angles
for angle in centered_list:
    # For each angle, create a dictionary to store the variables
    angle_data[angle] = {
        'xx_start': None,   # Replace None with actual data
        'xx_end': None,     # Replace None with actual data
        'dx': None,         # Replace None with actual data
        'dy': None,         # Replace None with actual data
        'yy_start': None,   # Replace None with actual data
        'yy_end': None,     # Replace None with actual data
        'controls': None    # Replace None with actual data
    }

## Filling in the data for each slit

In [ ]:
angle_data[148]['xx_start'] = 5
angle_data[148]['xx_end'] = 9
angle_data[148]['dx'] = 0.25
angle_data[148]['dy'] = 15
angle_data[148]['yy_start'] = 800
angle_data[148]['yy_end'] = 1100
angle_data[148]['controls'] = [7, 950]

angle_data[150]['xx_start'] = 5
angle_data[150]['xx_end'] = 9
angle_data[150]['dx'] = 0.25
angle_data[150]['dy'] = 15
angle_data[150]['yy_start'] = 600
angle_data[150]['yy_end'] = 1100
angle_data[150]['controls'] = [7, 1000]

angle_data[152]['xx_start'] = 5
angle_data[152]['xx_end'] = 9
angle_data[152]['dx'] = 0.25
angle_data[152]['dy'] = 15
angle_data[152]['yy_start'] = 650
angle_data[152]['yy_end'] = 1100
angle_data[152]['controls'] = [7, 850]

angle_data[154]['xx_start'] = 3
angle_data[154]['xx_end'] = 8.5
angle_data[154]['dx'] = 0.19
angle_data[154]['dy'] = 15
angle_data[154]['yy_start'] = 620
angle_data[154]['yy_end'] = 1000
angle_data[154]['controls'] = [5, 680]

angle_data[156]['xx_start'] = 4
angle_data[156]['xx_end'] = 9
angle_data[156]['dx'] = 0.2
angle_data[156]['dy'] = 15
angle_data[156]['yy_start'] = 620
angle_data[156]['yy_end'] = 1050
angle_data[156]['controls'] = [6, 750]

angle_data[158]['xx_start'] = 3
angle_data[158]['xx_end'] = 8.5
angle_data[158]['dx'] = 0.19
angle_data[158]['dy'] = 15
angle_data[158]['yy_start'] = 560
angle_data[158]['yy_end'] = 920
angle_data[158]['controls'] = [5, 700]

angle_data[160]['xx_start'] = 2
angle_data[160]['xx_end'] = 9
angle_data[160]['dx'] = 0.15
angle_data[160]['dy'] = 15
angle_data[160]['yy_start'] = 560
angle_data[160]['yy_end'] = 920
angle_data[160]['controls'] = [5, 700]

angle_data[162]['xx_start'] = 3
angle_data[162]['xx_end'] = 8
angle_data[162]['dx'] = 0.2
angle_data[162]['dy'] = 15
angle_data[162]['yy_start'] = 570
angle_data[162]['yy_end'] = 920
angle_data[162]['controls'] = [5, 670]

angle_data[164]['xx_start'] = 3
angle_data[164]['xx_end'] = 8
angle_data[164]['dx'] = 0.2
angle_data[164]['dy'] = 15
angle_data[164]['yy_start'] = 600
angle_data[164]['yy_end'] = 900
angle_data[164]['controls'] = [5, 650]

angle_data[166]['xx_start'] = 4
angle_data[166]['xx_end'] = 8
angle_data[166]['dx'] = 0.25
angle_data[166]['dy'] = 15
angle_data[166]['yy_start'] = 570
angle_data[166]['yy_end'] = 900
angle_data[166]['controls'] = [6, 650]

angle_data[168]['xx_start'] = 5
angle_data[168]['xx_end'] = 7
angle_data[168]['dx'] = 0.35
angle_data[168]['dy'] = 15
angle_data[168]['yy_start'] = 600
angle_data[168]['yy_end'] = 880
angle_data[168]['controls'] = [7, 700]

angle_data[170]['xx_start'] = 4
angle_data[170]['xx_end'] = 8
angle_data[170]['dx'] = 0.25
angle_data[170]['dy'] = 15
angle_data[170]['yy_start'] = 540
angle_data[170]['yy_end'] = 870
angle_data[170]['controls'] = [6, 640]

angle_data[172]['xx_start'] = 4
angle_data[172]['xx_end'] = 8
angle_data[172]['dx'] = 0.25
angle_data[172]['dy'] = 15
angle_data[172]['yy_start'] = 590
angle_data[172]['yy_end'] = 820
angle_data[172]['controls'] = [6, 590]

In [ ]:
# for angle in centered_list:
    
#     xx_start = angle_data[angle]['xx_start']
#     xx_end   = angle_data[angle]['xx_end']
#     dx       = angle_data[angle]['dx']
#     dy       = angle_data[angle]['dy']
#     yy_start = angle_data[angle]['yy_start']
#     yy_end   = angle_data[angle]['yy_end']
#     controls = angle_data[angle]['controls']
    
#     print(f'xx_start: {xx_start}')
#     print(f'xx_end: {xx_end}')
#     print(f'dx: {dx}\tdy: {dy}')
#     print(f'yy_start: {yy_start}')
#     print(f'yy_end: {yy_end}')
#     print(f'controls: {controls}\n')

In [ ]:
for i, angle in enumerate(centered_list):
    
    # Define start and end points and control point for the Bézier curve
    xx_start = angle_data[angle]['xx_start']
    xx_end   = angle_data[angle]['xx_end']
    dx       = angle_data[angle]['dx']
    dy       = angle_data[angle]['dy']
    yy_start = angle_data[angle]['yy_start']
    yy_end   = angle_data[angle]['yy_end']
    control  = angle_data[angle]['controls']
    
    X_start = generate_number_list(xx_start, dx, 2)
    X_end   = generate_number_list(xx_end, dx, 2)
    Y_start = generate_number_list(yy_start, dy, 2)
    Y_end   = generate_number_list(yy_end, dy, 2)
    
    height = output_obj[f'distances_{angle}deg'][0].copy()
    intens = output_obj[f'intensity_{angle}deg'][0].copy()
    
    # Define a psudo time and distance axes for the Bezier curve method to work
    psudo_time = np.arange(len(datenum_arr))
    psudo_dist = np.arange(len(height))
    
    x_coords_list = []
    y_coords_list = []
    
    fig = plt.figure(figsize=[8,6])
    ax = fig.add_subplot(111)
    ax.pcolormesh(psudo_time, psudo_dist, intens, vmin=0.5, vmax=1.7, cmap='Greys_r')
    
    # Extract values along the Bézier curves
    for (x1, y1, x2, y2) in zip(X_start, Y_start, X_end, Y_end):
        _, x_coords, y_coords = extract_bezier_values(intens, x1, y1, x2, y2, dx, control)
        x_coords_list.append(x_coords[:-1])
        y_coords_list.append(y_coords[:-1])
        ax.plot(x_coords[:-1], y_coords[:-1], 'ko--', linewidth=1, markersize=3, alpha=0.5)
    
    # Compute mean and standard error for x and y coordinates
    mean_x_coords, standard_error_x = compute_standard_error(x_coords_list)
    mean_y_coords, standard_error_y = compute_standard_error(y_coords_list)
    
    # Store the final curves values
    df = pd.DataFrame({
        'x_coords': mean_x_coords,
        'y_coords': mean_y_coords,
        'x_err': standard_error_x,
        'y_err': standard_error_y,
        'x1': xx_start,
        'x2': xx_end,
        'y1': yy_start,
        'y2': yy_end,
        'control_point_x': control[0],
        'control_point_y': control[1]
    })
    jmaps_coords_list[f'{angle}'].append(df)
    
    # Plot the mean Bezier curve with error bars
    ax.errorbar(mean_x_coords, mean_y_coords,
                xerr=standard_error_x, yerr=standard_error_y,
                fmt='ro--', ecolor='blue', barsabove=False, linewidth=2,
                markersize=4, elinewidth=1, capsize=3)
    
    ax.set_xlabel('Time step')
    ax.set_ylabel('Height step')
    ax.set_title(f"J-plot of {output_obj['instrument'][0]} Intensity Along Slit {angle} deg")
    ax.xaxis.set_minor_locator(AutoMinorLocator(n=2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(n=5))
    sol_limb = find_nearest_index(height, suvi_map.rsun_obs.value)
    ax.set_xlim(right=12)
    ax.set_ylim(bottom=sol_limb)
    plt.show()

### Export the tables and J-Maps with fitting for all slits

In [ ]:
df = jmaps_coords_list[f'{angle}'][0].copy()
df

In [ ]:
df

In [ ]:
height

In [ ]:
# Calculate the step size between consecutive elements
step_size = np.diff(height)
print(step_size)

In [ ]:
# If you want the average step size
average_step_size = np.mean(step_size)
print(average_step_size)

In [ ]:
y_coords

In [ ]:
y_coords * average_step_size

In [ ]:
sol_limb

In [ ]:
sol_limb_idx = (np.abs(height - sol_limb)).argmin()

In [ ]:
height[sol_limb_idx]

In [ ]:
suvi_map.rsun_obs.value

In [ ]:
for angle in centered_list:
    
    df = jmaps_coords_list[f'{angle}'][3].copy()

    height = output_obj[f'distances_{angle}deg'][0].copy()
    intens = output_obj[f'intensity_{angle}deg'][0].copy()
    
    # Extract time and distance arrays from the selected coordinates
    x_coords = df['x_coords'].astype('int')
    times_num = np.array(datenum_arr)[x_coords]
    df['time'] = times_num
    
    y_coords = df['y_coords']
    y_coords = round(y_coords).astype('int')
    distances = height[y_coords] # in arcsec
    df['distance'] = distances

    print(f'{times_num}\n{distances}\n\n')
    
    # Perform linear regression to fit a line
    slope, intercept, r_value, p_value, std_err = stats.linregress(times_num, distances)
    
    # get the radius of the solar disk
    sol_rad = const.equatorial_radius.to(u.km)
    
    # conversion factor from arcsec to km
    conversion_factor = sol_rad/output_obj['map_obj'][0].rsun_obs
    
    # convert distance from arcsec to km
    df['distance_km'] = df['distance'] * conversion_factor.value
    
    # calculate the distance difference in km
    df['distance_diff_km'] = df['distance_km'].diff()
    
    # convert time to datetime format
    df['datetime'] = [mdates.num2date(t) for t in df['time']]
    
    # calculate the time difference in seconds
    df['time_diff_s'] = df['datetime'].diff().dt.total_seconds()
    
    # calculate the speed in km/s
    df['speed_km_s'] = df['distance_diff_km'] / df['time_diff_s']

    # export data
    event_date = str(mdates.num2date(times_num[0]).date()).replace('-','')
    # df.to_csv(f"{savedir}/jplots/suvi/bezier/jmap_{output_obj['instrument'][0]}_{event_date}_slit_{angle}.csv")
    
    # # drop the first row which will have NaN values for the differences
    # df.dropna(inplace=True)
    
    # Calculate spline fit
    spline = UnivariateSpline(times_num, distances, k=1, s=None)  # s=0 for interpolation through all points
    
    # Generate the spline line
    spline_times = np.linspace(min(times_num), max(times_num), 1000)
    spline_distances = spline(spline_times)
    
    # Calculate the derivative of spline_distances with respect to spline_times
    spline_velocity = spline.derivative()(spline_times)
    
    # Convert velocity (in arcsec/day) to speed in km/s
    # 1 arcsec ≈ 733 km on the Sun's surface
    # 1 day = 86400 seconds
    speed_spline = spline_velocity * conversion_factor.value / 86400  # km/s
    
    # Calculate the polynomial fit
    polyfit_coeff = np.polyfit(times_num, distances, 2)  # Fit a 2nd order polynomial
    polyfit_line = np.polyval(polyfit_coeff, spline_times)
    
    # Calculate the derivative of the polynomial fit (velocity)
    polyfit_velocity = np.polyval(np.polyder(polyfit_coeff), spline_times)
    speed_polyfit = polyfit_velocity * conversion_factor.value / 86400  # km/s
    
    # plots the fitting lines
    fig = plt.figure(figsize=[8,6])
    ax = fig.add_subplot(111)
    ax.pcolormesh(datenum_arr, height, intens, vmin=0.5, vmax=1.7, cmap='Greys_r')

    # # Plot the mean Bezier curve with error bars
    # ax.errorbar(times_num, y_coords*average_step_size,
    #             xerr=df['x_err'],  yerr=df['y_err']*average_step_size,
    #             fmt='ro--', ecolor='blue', barsabove=False, linewidth=2,
    #             markersize=4, elinewidth=1, capsize=3)
    
    # Plot the fitted line
    fit_line = slope*times_num + intercept
    
    # Calculate the speed (slope in arcsec/day to speed in km/s)
    # 1 arcsec ≈ 733 km on the Sun's surface
    # 1 day = 86400 seconds
    speed_fit = slope * conversion_factor.value/86400  # km/s
    speed = df['speed_km_s'].mean()
    
    # Plot the selected points
    ax.plot(times_num, distances, 'ro', label=f'Mean speed: {speed:.2f} km/s')
    ax.plot(times_num, fit_line, ls='--', color='yellow',
            label=f'Linear fit: {speed_fit:.2f} km/s')
    
    # Plot the spline fit line
    ax.plot(spline_times, spline_distances, ls='--', color='tab:red',
            label=f'Spline fit: {np.nanmean(speed_spline):.2f} km/s')
    
    # Plot the polynomial fit line
    ax.plot(spline_times, polyfit_line, ls='-', color='tab:blue',
               label=f'2nd-order Polynomial fit: {np.nanmean(speed_polyfit):.2f} km/s')
    
    ax.set_xlabel('Time (UT)')
    ax.set_ylabel('Angular distance (arcsec)')
    ax.set_title(f"J-plot of {output_obj['instrument'][0]} Intensity Along Slit {angle} deg")
    ax.legend(loc='lower right')
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_minor_locator(AutoMinorLocator(n=2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(n=2))
    ax.set_xlim(right=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:55:00"))
    ax.set_ylim(bottom=suvi_map.rsun_obs.value)
    # fig.savefig(f"{savedir}/jplots/suvi/bezier/jmap_{output_obj['instrument'][0]}_{event_date}_slit_{angle}.png", format='png', bbox_inches='tight')
    plt.show()

### Doing all the slits one by one because the for-loop didn't work ...

In [ ]:
# angle = 172

# height = output_obj[f'distances_{angle}deg'][0].copy()
# intens = output_obj[f'intensity_{angle}deg'][0].copy()

# print(f'Working on slit: {angle} deg.')

# # Number of repetitions
# num_repeats = 5
# current_trial = 0

# # Dictionary to store coordinates for each trial
# feature_coords_slit = {f'trial_{i}': [] for i in range(num_repeats)}

# # Text handle to update on the plot
# text_handle = None

# # plot the j-map
# fig = plt.figure(figsize=[10,7])
# ax = fig.add_subplot(111)
# plt.ion()
# ax.pcolormesh(datenum_arr, height, intens,
#               vmin=0.5, vmax=1.7,
#               cmap='Greys_r')
# ax.set_xlabel('Time (UT)')
# ax.set_ylabel('Angular distance (arcsec)')
# ax.set_title(f"J-plot of {output_obj['instrument'][0]} Intensity Along Slit {angle} deg")
# ax.xaxis_date()
# ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# ax.xaxis.set_minor_locator(AutoMinorLocator(n=5))
# ax.yaxis.set_minor_locator(AutoMinorLocator(n=5))
# ax.set_xlim(left=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:10:00"),
#             right=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:55:00"))
# ax.set_ylim(bottom=m.rsun_obs.value)

# # Connect the click event to the onclick function
# cid = fig.canvas.mpl_connect('button_press_event', onclick)
# plt.show(block=False)
# plt.pause(0.001)

In [ ]:
# calc the standard error
mean_values, standard_errors = compute_standard_error(feature_coords_slit)

x_mean, y_mean = zip(*mean_values)
x_err, y_err = zip(*standard_errors)

event_date = output_obj['time'][0].split(' ')[0].replace('-','')
output_filename = f"jmap_{output_obj['instrument'][0]}_{event_date}_slit_{angle}"

df = pd.DataFrame({'time': x_mean, 'distance': y_mean})
df.to_pickle(f'{savedir}/jplots/suvi/clicking/coords_{output_filename}.pkl')

# Extract time and distance arrays from the selected coordinates
times_num, distances = df['time'].values, df['distance'].values

# Perform linear regression to fit a line
slope, intercept, r_value, p_value, std_err = stats.linregress(times_num, distances)

# get the radius of the solar disk
sol_rad = const.equatorial_radius.to(u.km)

# conversion factor from arcsec to km
conversion_factor = sol_rad/output_obj['map_obj'][0].rsun_obs

# convert distance from arcsec to km
df['distance_km'] = distances * conversion_factor.value

# calculate the distance difference in km
df['distance_diff_km'] = df['distance_km'].diff()

# convert time to datetime format
df['datetime'] = [mdates.num2date(t) for t in times_num]

# calculate the time difference in seconds
df['time_diff_s'] = df['datetime'].diff().dt.total_seconds()

# calculate the speed in km/s
df['speed_km_s'] = df['distance_diff_km'] / df['time_diff_s']

df.to_csv(f'{savedir}/jplots/suvi/clicking/{output_filename}.csv')
print(f'J-plot data saved at: {savedir}/jplots/suvi/clicking/{output_filename}.csv')

# # drop the first row which will have NaN values for the differences
# df.dropna(inplace=True)

# Calculate spline fit
spline = UnivariateSpline(times_num, distances, k=1, s=None) # s=0 for interpolation through all points

# Generate the spline line
spline_times = np.linspace(min(times_num), max(times_num), 1000)
spline_distances = spline(spline_times)

# Calculate the derivative of spline_distances with respect to spline_times
spline_velocity = spline.derivative()(spline_times)

# Convert velocity (in arcsec/day) to speed in km/s
# 1 arcsec ≈ 733 km on the Sun's surface
# 1 day = 86400 seconds
speed_spline = spline_velocity * conversion_factor.value / 86400  # km/s

# Calculate the polynomial fit
polyfit_coeff = np.polyfit(times_num, distances, 2)  # Fit a 2nd order polynomial
polyfit_line = np.polyval(polyfit_coeff, spline_times)

# Calculate the derivative of the polynomial fit (velocity)
polyfit_velocity = np.polyval(np.polyder(polyfit_coeff), spline_times)
speed_polyfit = polyfit_velocity * conversion_factor.value / 86400  # km/s

# show the final J-map with the fitting lines and speed estimations
fig = plt.figure(figsize=[8,6])
ax = fig.add_subplot(111)
ax.pcolormesh(datenum_arr, height, intens, vmin=0.5, vmax=1.7, cmap='Greys_r')

# Plot the fitted line
fit_line = slope*times_num + intercept

# Calculate the speed (slope in arcsec/day to speed in km/s)
speed_fit = slope * conversion_factor.value/86400  # km/s
speed = df['speed_km_s'].mean()

# Plot the selected points with error bars
ax.errorbar(x_mean, y_mean, xerr=x_err, yerr=y_err,
            fmt='ro', ecolor='blue', barsabove=False,
            markersize=4, elinewidth=1, capsize=3, label=f'Mean speed: {speed:.2f} km/s')

ax.plot(times_num, fit_line, ls='--', color='yellow',
        label=f'Linear fit: {speed_fit:.2f} km/s')

# Plot the spline fit line
ax.plot(spline_times, spline_distances, ls='--', color='tab:red',
        label=f'Spline fit: {np.nanmean(speed_spline):.2f} km/s')

# Plot the polynomial fit line
ax.plot(spline_times, polyfit_line, ls='-', color='tab:blue',
           label=f'2nd-order Polynomial fit: {np.nanmean(speed_polyfit):.2f} km/s')

ax.set_xlabel('Time (UT)')
ax.set_ylabel('Angular distance (arcsec)')
ax.set_title(f"J-plot of {output_obj['instrument'][0]} Intensity Along Slit {angle} deg.")
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.legend(loc='lower right')
ax.set_xlim(left=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:10:00"),
            right=pd.Timestamp(f"{output_obj['time'][0].split(' ')[0]} 17:55:00"))
ax.set_ylim(bottom=m.rsun_obs.value)
fig.tight_layout()
fig.savefig(f'{savedir}/jplots/suvi/clicking/{output_filename}.png',
            format='png', bbox_inches='tight')
print(f'Figure: {output_filename} has been exported.\n')
plt.show()